In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")



In [ ]:
previous_application_df.head(5)

In [ ]:
#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))


In [ ]:

previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(3)

In [ ]:
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
dtale.show(df_wide)

In [ ]:
previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
previous_application_df= pd.get_dummies(previous_application_df, columns=["name_contract_status"])
dtale.show(previous_application_df)

In [ ]:

previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p(previous_application_df["total_interest_charged"])



agg_metrics_df= previous_application_df.groupby("id_curr").agg({

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],

    #others_monetary
    "diff_application_credit": ["max","mean","min","median","sum"],
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","std","min","median"],
    "ratio_credit_to_goods" : ["max","mean","std","min","median"],
    "ratio_credit_to_annuity" : ["max","mean","std","min","median"],

    #categoricals
    "name_contract_status_approved": ["mean","sum"],
    "name_contract_status_canceled": ["mean","sum"],
    "name_contract_status_refused": ["mean","sum"],
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","min","max","sum"]
})




In [ ]:
agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]
agg_metrics_df= agg_metrics_df.reset_index()

In [ ]:
agg_metrics_df.rename(columns={"id_prev_count": "applications_count"})
agg_metrics_df.head()


In [ ]:
previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")

In [ ]:
previous_application_ready_to_merge.head(5)
previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "previous_application_ready_to_join.parquet")
